# Daily limit strategy parameter comparison

Compare fixed discounts and expiry horizons using terminal value, drawdown, and Calmar ratio.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import daily_data_summary, load_or_fetch_twelve_data_daily

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

from retail_sp500.limit_orders import evaluate_limit_discount_grid
from retail_sp500.limit_plotting import calmar_by_discount_figure, return_drawdown_figure
from retail_sp500.limit_portfolio import evaluate_recurring_limit_grid

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

source = daily_data_summary(daily, symbol=SYMBOL)
print(source)
assert source["interval"] == "1day"
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

## Build the strategy grid

In [ ]:
DISCOUNTS = np.arange(0.0, 0.0501, 0.001)
WAIT_HORIZONS = [1, 3, 5, 10, 20]

one_session = evaluate_limit_discount_grid(daily, DISCOUNTS)
strategy_grid = pd.concat(
    [
        evaluate_recurring_limit_grid(
            daily,
            DISCOUNTS,
            max_wait_sessions=wait,
            initial_cash=100_000,
            monthly_contribution=1_000,
        ).assign(wait_horizon=wait)
        for wait in WAIT_HORIZONS
    ],
    ignore_index=True,
)

strategy_grid.sort_values("calmar_ratio", ascending=False).head(20)

## Best strategy under different objectives

In [ ]:
best_terminal = strategy_grid.loc[
    strategy_grid.groupby("wait_horizon")["ending_excess_value"].idxmax()
].sort_values("wait_horizon")

best_calmar = strategy_grid.dropna(subset=["calmar_ratio"]).loc[
    strategy_grid.dropna(subset=["calmar_ratio"]).groupby("wait_horizon")["calmar_ratio"].idxmax()
].sort_values("wait_horizon")

pd.concat({"Best terminal value": best_terminal, "Best Calmar": best_calmar})[[
    "wait_horizon",
    "discount",
    "annualized_return",
    "max_drawdown",
    "calmar_ratio",
    "ending_excess_value",
    "limit_fill_rate",
    "forced_fill_rate",
]]

In [ ]:
px.line(
    strategy_grid,
    x="discount",
    y="ending_excess_pct_of_contributions",
    color="wait_horizon",
    title="Terminal execution advantage by strategy",
    labels={
        "discount": "Limit below preceding close",
        "ending_excess_pct_of_contributions": "Excess ending value / contributions",
        "wait_horizon": "Maximum wait sessions",
    },
).update_xaxes(tickformat=".1%").update_yaxes(tickformat=".1%").show()

In [ ]:
calmar_by_discount_figure(strategy_grid).show()

In [ ]:
return_drawdown_figure(strategy_grid).show()

In [ ]:
px.line(
    strategy_grid,
    x="discount",
    y="forced_fill_rate",
    color="wait_horizon",
    title="Forced execution rate by strategy",
    labels={
        "discount": "Limit below preceding close",
        "forced_fill_rate": "Forced fill rate",
        "wait_horizon": "Maximum wait sessions",
    },
).update_xaxes(tickformat=".1%").update_yaxes(tickformat=".1%").show()

## Interpretation boundary

Calmar rewards return relative to the worst observed drawdown. It does not capture every risk and remains sample-dependent. Dividends, cash yield, spreads, fees, taxes, and SGD/USD effects are excluded.